In [2]:
import folium
import pandas as pd
import matplotlib


stations = [
    {"name": "Qods (Malek Shahr)", "lat": 32.71472, "lon": 51.60472},
    {"name": "Baharestan",          "lat": 32.71528, "lon": 51.62528},
    {"name": "Golestan",            "lat": 32.71528, "lon": 51.62722},
    {"name": "Shahid Mofateh",      "lat": 32.71667, "lon": 51.64528},
    {"name": "Shahid Alikhani",     "lat": 32.71611, "lon": 51.66083},
    {"name": "Jaber",               "lat": 32.70889, "lon": 51.66722},
    {"name": "Kaveh",               "lat": 32.69806, "lon": 51.67417},
    {"name": "Shahid Chamran",      "lat": 32.68750, "lon": 51.67583},
    {"name": "Shahid Bahonar",      "lat": 32.67889, "lon": 51.67556},
    {"name": "Shohada",             "lat": 32.67222, "lon": 51.67250},
    {"name": "Takhti",              "lat": 32.66537, "lon": 51.67044},
    {"name": "Emam Hossein",        "lat": 32.65785, "lon": 51.66956},
    {"name": "Enqelab",             "lat": 32.64944, "lon": 51.66840},
    {"name": "Si-o-se Pol",         "lat": 32.63929, "lon": 51.66670},
    {"name": "Shari'ati",           "lat": 32.62830, "lon": 51.66550},
    {"name": "Azadi",               "lat": 32.62236, "lon": 51.66483},
    {"name": "Daneshgah-e Esfahan", "lat": 32.61444, "lon": 51.66371},
    {"name": "Kargar",              "lat": 32.60694, "lon": 51.66376},
    {"name": "Kuy-e Emam",          "lat": 32.59823, "lon": 51.66861},
    {"name": "Defa'-e Moqaddas (Soffeh)", "lat": 32.58983, "lon": 51.67031},
]

stations_df = pd.DataFrame(stations)
stations_df


,name,lat,lon
0,Qods (Malek Shahr),32.71472,51.60472
1,Baharestan,32.71528,51.62528
2,Golestan,32.71528,51.62722
3,Shahid Mofateh,32.71667,51.64528
4,Shahid Alikhani,32.71611,51.66083
5,Jaber,32.70889,51.66722
6,Kaveh,32.69806,51.67417
7,Shahid Chamran,32.68750,51.67583
8,Shahid Bahonar,32.67889,51.67556
9,Shohada,32.67222,51.67250


In [ ]:
# District population
district_population = {
    1: 80807, 2: 76733, 3: 111013, 4: 147139, 5: 132984,
    6: 112896, 7: 206588, 8: 243563, 9: 78271, 10: 200702,
    11: 58334, 12: 155413, 13: 164413, 14: 160354, 15: 138247,
}

In [3]:
import requests
import re
import folium
from folium.plugins import MarkerCluster

bbox = "32.40,51.40,32.90,52.00"
overpass_url = "https://overpass-api.de/api/interpreter"

# 1. Query structured OSM tags for major facilities and bases
query = f"""
[out:json][timeout:90];
(
  // Verified military sites, bases, barracks, bunkers, airfields
  nwr["military"~"base|airfield|barracks|bunker|range|training_area"]({bbox});
  nwr["landuse"="military"]({bbox});

  // Major aerodromes / runways
  nwr["aeroway"~"aerodrome|runway"]({bbox});

  // Dedicated nuclear industrial nodes
  nwr["industrial"="nuclear"]({bbox});
  nwr["plant:source"="nuclear"]({bbox});

  // Heavy energy generation & large-scale refining
  nwr["power"="plant"]({bbox});
  nwr["man_made"="petroleum_refinery"]({bbox});
  nwr["industrial"~"oil|petrochemical|steel|heavy"]({bbox});

  // Named industrial parks and complexes (boundaries & areas)
  nwr["landuse"="industrial"]["name"~"شهرک صنعتی"]({bbox});
  
  // Basij / IRGC stations (queried with explicit word boundaries)
  nwr["amenity"="military"]({bbox});
  nwr["office"="military"]({bbox});
  nwr["name"~"(^|\\s)(بسیج|حوزه مقاومت|سپاه|پادگان)($|\\s)"]({bbox});
);
out center tags;
"""

headers = {
    'User-Agent': 'IsfahanOptimizationResearch/3.0 (academic)'
}

response = requests.post(overpass_url, data={'data': query}, headers=headers)
data = response.json()

# 2. Strict False-Positive Filtering
# Discard retail, civic streets, and regional "Sepahan" commercial noise
STRICT_DROP_TERMS = [
    "سپاهان", "سپاهانشهر", "سپاهان‌شهر", "میدان", "بزرگراه", "جاده", "گذر", 
    "خیابان", "کوچه", "نان", "سوپر", "فروشگاه", "تعمیرگاه", "بوستان", "پارکینگ",
    "تصویربرداری", "درمانگاه", "کلینیک", "مطب", "داروخانه", "دندان", "املاک", 
    "پاساژ", "بازار", "مدرسه", "دبیرستان", "بانک", "پست", "شیرینی", "رستوران",
    "کافه", "استخر", "باشگاه", "سرویس خواب", "پروتئین", "چوب", "پلاستیک", "بوتیک"
]

def is_valid_strategic_target(tags):
    name = tags.get('name', '')
    military = tags.get('military', '')
    aeroway = tags.get('aeroway', '')
    power = tags.get('power', '')
    man_made = tags.get('man_made', '')
    landuse = tags.get('landuse', '')
    industrial = tags.get('industrial', '')

    # Reject non-strategic commercial/transport tags
    if tags.get('highway') or tags.get('shop') or tags.get('amenity') in ['pharmacy', 'restaurant', 'cafe', 'bank', 'post_office']:
        return False

    # Filter out false positive name matches
    for drop_term in STRICT_DROP_TERMS:
        if drop_term in name:
            # Keep only if it is explicitly an industrial oil refinery
            if drop_term == "سپاهان" and "پالایشگاه" in name:
                return True
            return False

    # Ensure it qualifies as one of the desired core assets
    is_defense = military or landuse == 'military' or any(k in name for k in ["پادگان", "بسیج", "لشکر", "تیپ", "هوایی", "توپخانه"])
    is_aviation = aeroway in ['aerodrome', 'runway']
    is_nuclear = industrial == 'nuclear' or "هسته‌ای" in name or "هسته ای" in name
    is_energy = power == 'plant' or man_made == 'petroleum_refinery' or any(k in name for k in ["پالایشگاه", "پتروشیمی", "نیروگاه"])
    is_heavy_park = "شهرک صنعتی" in name or (landuse == 'industrial' and industrial in ['steel', 'heavy', 'metal'])

    return is_defense or is_aviation or is_nuclear or is_energy or is_heavy_park

def assign_category(tags):
    name = tags.get('name', '')
    if tags.get('industrial') == 'nuclear' or "هسته‌ای" in name or "هسته ای" in name:
        return "Nuclear"
    if tags.get('aeroway') in ['aerodrome', 'runway'] or "پایگاه هوایی" in name or "فرودگاه" in name:
        return "Aviation & Air Bases"
    if tags.get('power') == 'plant' or tags.get('man_made') == 'petroleum_refinery' or any(k in name for k in ["پالایشگاه", "پتروشیمی", "نیروگاه"]):
        return "Energy & Refining"
    if "شهرک صنعتی" in name or tags.get('industrial') in ['steel', 'heavy']:
        return "Heavy Industrial Parks"
    return "Defense & Military"

refined_targets = []
seen = set()

for el in data.get('elements', []):
    tags = el.get('tags', {})
    if not is_valid_strategic_target(tags):
        continue

    lat = el.get('lat') or el.get('center', {}).get('lat')
    lon = el.get('lon') or el.get('center', {}).get('lon')
    name = tags.get('name:en') or tags.get('name', 'Military/Defense Facility')
    
    coord_key = (round(lat, 4), round(lon, 4))
    if coord_key in seen:
        continue
    seen.add(coord_key)

    refined_targets.append({
        'id': el.get('id'),
        'name': name,
        'category': assign_category(tags),
        'type': tags.get('military') or tags.get('power') or tags.get('industrial') or tags.get('aeroway') or 'complex',
        'lat': lat,
        'lon': lon
    })

print(f"Retrieved {len(refined_targets)} valid strategic targets.")

Retrieved 150 valid strategic targets.


In [5]:
import numpy as np

arr = np.array(refined_targets)

# Save single array
np.save('my_array.npy', arr)

In [54]:
m = folium.Map(
    location=[32.6546, 51.6680],
    zoom_start=11,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}',
    attr='ESRI'
)

# Metro Alignment & Shelters
subway_layer = folium.FeatureGroup(name="Subway Stations (Safe Shelters)")
folium.PolyLine(
    locations=[[s["lat"], s["lon"]] for s in stations],
    color="#2563EB", weight=4, opacity=0.85
).add_to(subway_layer)

for s in stations:
    folium.CircleMarker(
        location=[s["lat"], s["lon"]],
        radius=6, color="#1E40AF", fill=True, fill_color="#3B82F6", fill_opacity=0.95,
        popup=f"<b>Shelter (Metro):</b> {s['name']}", tooltip=s['name']
    ).add_to(subway_layer)

# Categorized High-Value Targets
targets_layer = MarkerCluster(name="Verified Strategic Targets")

CATEGORY_COLORS = {
    "Nuclear":                  {"color": "#78350F", "fill": "#F59E0B", "radius": 7},
    "Defense & Military":       {"color": "#991B1B", "fill": "#EF4444", "radius": 6},
    "Aviation & Air Bases":     {"color": "#4C1D95", "fill": "#8B5CF6", "radius": 6},
    "Energy & Refining":        {"color": "#C2410C", "fill": "#FB923C", "radius": 6},
    "Heavy Industrial Parks":   {"color": "#1F2937", "fill": "#6B7280", "radius": 5}
}

for t in refined_targets:
    style = CATEGORY_COLORS.get(t["category"])
    folium.CircleMarker(
        location=[t["lat"], t["lon"]],
        radius=style["radius"],
        color=style["color"],
        fill=True,
        fill_color=style["fill"],
        fill_opacity=0.85,
        popup=folium.Popup(
            f"<b>{t['name']}</b><br>"
            f"<b>Category:</b> {t['category']}<br>"
            f"<b>Type:</b> {t['type']}",
            max_width=250
        ),
        tooltip=t['name']
    ).add_to(targets_layer)

subway_layer.add_to(m)
targets_layer.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

m.save("isfahan_strategic_map.html")

In [ ]:
import webbrowser
webbrowser.open("isfahan_districts.html")

True

In [73]:
import json
import folium
from folium.plugins import MarkerCluster

# 1. Load the GeoJSON file
with open("isfahan_raw.geojson", "r", encoding="utf-8") as f:
    raw_geojson = json.load(f)

# 2. Population mapping
district_population = {
    1: 80807, 2: 76733, 3: 111013, 4: 147139, 5: 132984,
    6: 112896, 7: 206588, 8: 243563, 9: 78271, 10: 200702,
    11: 58334, 12: 155413, 13: 164413, 14: 160354, 15: 138247
}

persian_to_eng = {
    '۱': 1, '۲': 2, '۳': 3, '۴': 4, '۵': 5, '۶': 6, '۷': 7, '۸': 8,
    '۹': 9, '۱۰': 10, '۱۱': 11, '۱۲': 12, '۱۳': 13, '۱۴': 14, '۱۵': 15
}

# 3. Filter strictly for the 15 Isfahan municipal polygons
clean_features = []
seen_ids = set()

for feat in raw_geojson.get("features", []):
    geom_type = feat.get("geometry", {}).get("type")
    props = feat.get("properties", {})
    name = props.get("name", "")
    admin_level = str(props.get("admin_level", ""))

    if geom_type not in ["Polygon", "MultiPolygon"] or admin_level != "9" or "خمینی" in name:
        continue

    d_num = None
    for p_num, val in persian_to_eng.items():
        if f"منطقه {p_num}" == name.strip() or f"منطقه {p_num} " in name:
            d_num = val
            break

    if d_num and d_num not in seen_ids:
        seen_ids.add(d_num)
        pop = district_population.get(d_num, 0)
        feat["properties"]["district_id"] = d_num
        feat["properties"]["population"] = f"{pop:,}"
        feat["properties"]["label"] = f"منطقه {d_num:02d}"
        clean_features.append(feat)

isfahan_districts_geojson = {
    "type": "FeatureCollection",
    "features": clean_features
}

# 4. Color Palette
min_pop = min(district_population.values())
max_pop = max(district_population.values())

def get_choropleth_color(pop_val):
    norm = (pop_val - min_pop) / (max_pop - min_pop)
    r = int(235 - norm * (235 - 110))
    g = int(175 - norm * (175 - 35))
    b = int(170 - norm * (170 - 35))
    return f"#{r:02x}{g:02x}{b:02x}"

def style_function(feature):
    d_id = feature["properties"]["district_id"]
    pop = district_population.get(d_id, min_pop)
    return {
        "fillColor": get_choropleth_color(pop),
        "color": "#333333",
        "weight": 1.5,
        "fillOpacity": 0.70
    }

# 5. Initialize Map with Free ESRI Canvas (No API key required)
m = folium.Map(
    location=[32.6546, 51.6680],
    zoom_start=11,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ"
)

districts_layer = folium.FeatureGroup(name="Isfahan Districts (مناطق ۱۵ گانه)")
folium.GeoJson(
    isfahan_districts_geojson,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=["label", "population"],
        aliases=["District:", "Population:"],
        localize=True
    ),
    popup=folium.GeoJsonPopup(
        fields=["label", "population"],
        aliases=["نام منطقه:", "جمعیت:"]
    )
).add_to(districts_layer)
districts_layer.add_to(m)

# Subway Line & Shelters
subway_layer = folium.FeatureGroup(name="Subway Stations (Shelters)")
folium.PolyLine([[s["lat"], s["lon"]] for s in stations], color="#1D4ED8", weight=4, opacity=0.9).add_to(subway_layer)
for s in stations:
    folium.CircleMarker(
        location=[s["lat"], s["lon"]], radius=6, color="#1E3A8A", fill=True,
        fill_color="#3B82F6", fill_opacity=0.95, popup=s["name"], tooltip=s["name"]
    ).add_to(subway_layer)
subway_layer.add_to(m)

# Strategic Targets
targets_layer = MarkerCluster(name="Verified Strategic Targets")
for t in refined_targets:
    folium.CircleMarker(
        location=[t["lat"], t["lon"]], radius=5, color="#7F1D1D", fill=True,
        fill_color="#DC2626", fill_opacity=0.85, popup=f"<b>{t['name']}</b><br>{t['category']}"
    ).add_to(targets_layer)
targets_layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.save("isfahan_districts.html")